# 🧲 vinamax — симулятор магнитных наночастиц

**Решает уравнение Ландау-Лифшица-Гильберта (LLG)** для ансамбля наночастиц:
- Распределение по размерам (логнормальное)
- Смешанный ансамбль: **Blocked** (LLG + тепловой шум) + **SP** (прыжки Нееля)
- Поддержка единиц **СИ** и **СГС**
- Демагнетизирующее взаимодействие (диполь-диполь)


## Шаг 1 — Загрузить vinamax.py

In [ ]:
from google.colab import files
uploaded = files.upload()   # ← выбрать vinamax.py

## Шаг 2 — Импорт и настройка

In [ ]:
from vinamax import (
    Simulation, ParticleType, SizeDistribution,
    blocking_radius, neel_time
)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('✓ Импорт успешен')

## Шаг 3 — Параметры симуляции
Выберите систему единиц: **`'SI'`** или **`'CGS'`**

In [ ]:
# ── Выберите систему единиц ──────────────────────────────
UNITS = 'CGS'   # 'SI' или 'CGS'

# ── Параметры материала (Fe₃O₄) ─────────────────────────
if UNITS == 'CGS':
    MSAT   = 480        # эму/см³
    KU1    = 1.1e5      # эрг/см³
    B_APPL = 50.0       # Oe вдоль Z
    print(f'СГС: Msat={MSAT} эму/см³, Ku={KU1:.1e} эрг/см³, B={B_APPL} Oe')
else:
    MSAT   = 480e3      # А/м
    KU1    = 1.1e4      # Дж/м³
    B_APPL = 5e-3       # Тл вдоль Z
    print(f'СИ:  Msat={MSAT:.0f} А/м, Ku={KU1:.1e} Дж/м³, B={B_APPL*1e3:.1f} мТл')

# ── Параметры частиц ─────────────────────────────────────
R_MEDIAN  = 8.0    # нм  (медианный радиус)
SIGMA_GEO = 1.4    # геометрическое ст. откл.
COATING   = 2.0    # нм  (оболочка)
N_PART    = 50     # количество частиц

# ── Параметры симуляции ──────────────────────────────────
ALPHA     = 0.1
TEMP_K    = 300.   # К
DT        = 5e-12  # с
DURATION  = 2e-9   # с
SIGMA_THR = 3.0    # порог Blocked/SP

print(f'\nЧастицы: N={N_PART}, r_med={R_MEDIAN} нм, σ={SIGMA_GEO}')
print(f'α={ALPHA}, T={TEMP_K} К, Δt={DT:.0e} с, длительность={DURATION*1e9:.1f} нс')

## Шаг 4 — Создать и запустить симуляцию

In [ ]:
sim = Simulation(units=UNITS)
sim.set_random_seed(42)

# Мир 600×600×600 нм
sim.world(0, 0, 0, 600)

# Физика
sim.Alpha     = ALPHA
sim.Temp      = TEMP_K
sim.Tau0      = 1e-9
sim.Dt        = DT
sim.T         = 0.
sim.Brown     = True
sim.Jumpnoise = True
sim.Demag     = True

if UNITS == 'CGS':
    sim.ku1 = KU1          # свойство с авто-конверсией
else:
    sim.Ku1 = KU1

# Частицы с логнормальным распределением
sim.add_particles_lognormal(
    n=N_PART, r_median=R_MEDIAN, sigma_geo=SIGMA_GEO,
    r_h_coating=COATING, seed=42
)
sim.msat(MSAT)
sim.anisotropy_axis_random()
sim.m_random()

# Внешнее поле
if UNITS == 'CGS':
    sim.B_ext = sim.units.wrap_B_ext(lambda t: (0., 0., B_APPL))
else:
    sim.B_ext = lambda t: (0., 0., B_APPL)

# Классификация
stats = sim.classify_particles(sigma_threshold=SIGMA_THR)
r_crit = sim.units.radius_from_si(blocking_radius(sim.Ku1, sim.Temp, tau0=sim.Tau0))
print(f'r_crit = {r_crit:.2f} нм')
print(f'Blocked: {stats["n_blocked"]}  SP: {stats["n_sp"]}  (всего {stats["n_total"]})')

# Запуск
sim.set_solver('rk4')
sim.output(50e-12, 'result.txt')
print(f'\nЗапуск {DURATION*1e9:.1f} нс...')
sim.run(DURATION)

m_avg = sim.give_m()
E     = sim.E_total()
print(f'\n✓ Готово!')
print(f'  <mx>={m_avg[0]:.4f}  <my>={m_avg[1]:.4f}  <mz>={m_avg[2]:.4f}')
print(f'  E_total = {E:.4e} {sim.units.lbl_energy}')

## Шаг 5 — Визуализация

### 📊 График намагниченности vs время

In [ ]:
data = np.loadtxt('result.txt', comments='#')
t  = data[:, 0] * 1e9   # в нс
mx, my, mz = data[:, 1], data[:, 2], data[:, 3]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, mx, color='#E63946', lw=1.8, label='$\\langle m_x \\rangle$')
ax.plot(t, my, color='#2A9D8F', lw=1.8, label='$\\langle m_y \\rangle$')
ax.plot(t, mz, color='#457B9D', lw=1.8, label='$\\langle m_z \\rangle$')
ax.axhline(0, color='gray', lw=0.6, ls='--')
ax.set_xlabel('Время (нс)', fontsize=12)
ax.set_ylabel('m', fontsize=12)
ax.set_title('Динамика намагниченности', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(-1.05, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### 📏 Распределение размеров частиц

In [ ]:
radii_nm = np.array([sim.units.radius_from_si(p.r) for p in sim.particles])
types    = np.array([p.ptype.value for p in sim.particles])
r_bl = radii_nm[types == 'blocked']
r_sp = radii_nm[types == 'sp']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Гистограмма
bins = np.linspace(radii_nm.min()*0.8, radii_nm.max()*1.1, 22)
if len(r_bl): axes[0].hist(r_bl, bins=bins, color='#457B9D', alpha=0.85,
                            label=f'Blocked ({len(r_bl)})', edgecolor='white', lw=0.5)
if len(r_sp): axes[0].hist(r_sp, bins=bins, color='#E63946', alpha=0.85,
                            label=f'SP ({len(r_sp)})', edgecolor='white', lw=0.5)
axes[0].axvline(r_crit, color='#F4A261', lw=2.2, ls='--',
                label=f'$r_{{crit}}$ = {r_crit:.1f} нм')

# Логнормальная теоретическая кривая
from vinamax import SizeDistribution
dist = SizeDistribution(sim.units.radius_to_si(R_MEDIAN), SIGMA_GEO)
r_arr = np.linspace(radii_nm.min()*0.5, radii_nm.max()*1.3, 200)
r_arr_si = r_arr * 1e-9
pdf = dist.pdf(r_arr_si)
pdf_norm = pdf / pdf.max() * len(radii_nm) * (bins[1]-bins[0])
axes[0].plot(r_arr, pdf_norm, 'k--', lw=1.5, label='Логнорм. распр.')

axes[0].set_xlabel(f'Радиус ({sim.units.lbl_radius})', fontsize=11)
axes[0].set_ylabel('N', fontsize=11)
axes[0].set_title('Распределение по размерам', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.25)

# Круговая
n_bl, n_sp_ = len(r_bl), len(r_sp)
total = n_bl + n_sp_
if total > 0:
    axes[1].pie(
        [n_bl, n_sp_],
        labels=[f'Blocked\n{n_bl} ({100*n_bl//total}%)',
                f'SP\n{n_sp_} ({100*n_sp_//total}%)'],
        colors=['#457B9D', '#E63946'],
        startangle=90,
        wedgeprops=dict(width=0.55, edgecolor='white', lw=2)
    )
axes[1].set_title('Состав ансамбля', fontsize=12, fontweight='bold')

fig.suptitle(
    f'N={len(sim.particles)}, r_med={R_MEDIAN} нм, σ={SIGMA_GEO}, T={TEMP_K} K',
    fontsize=11
)
fig.tight_layout()
plt.show()

### 🌐 3D-расположение частиц

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(111, projection='3d')

color_map = {'blocked': '#457B9D', 'sp': '#E63946', 'auto': '#888'}

for p in sim.particles:
    x = sim.units.radius_from_si(p.x)
    y = sim.units.radius_from_si(p.y)
    z = sim.units.radius_from_si(p.z)
    r = sim.units.radius_from_si(p.r)
    col = color_map.get(p.ptype.value, '#888')

    ax.scatter(x, y, z, s=max((r*3)**2, 10),
               color=col, alpha=0.65, edgecolors='white', lw=0.4)

    # стрелка намагниченности
    sc = r * 2.0
    ax.quiver(x, y, z,
              p.m[0]*sc, p.m[1]*sc, p.m[2]*sc,
              color='black', lw=0.7, arrow_length_ratio=0.35, alpha=0.75)

lbl = sim.units.lbl_radius
ax.set_xlabel(f'X ({lbl})', fontsize=10)
ax.set_ylabel(f'Y ({lbl})', fontsize=10)
ax.set_zlabel(f'Z ({lbl})', fontsize=10)
ax.set_title('3D-расположение частиц (стрелки = m)', fontsize=12, fontweight='bold')

handles = [
    mpatches.Patch(color='#457B9D', label=f'Blocked ({sim._n_blocked})'),
    mpatches.Patch(color='#E63946', label=f'SP ({sim._n_sp})')
]
ax.legend(handles=handles, fontsize=10)
fig.tight_layout()
plt.show()

### 📋 Сводная панель (Dashboard)

In [ ]:
fig = plt.figure(figsize=(15, 9))
gspec = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# ── 1. mx, my, mz vs t ───────────────────────────────────────────────────────
ax1 = fig.add_subplot(gspec[0, :2])
ax1.plot(t, mx, color='#E63946', lw=1.8, label='$\\langle m_x\\rangle$')
ax1.plot(t, my, color='#2A9D8F', lw=1.8, label='$\\langle m_y\\rangle$')
ax1.plot(t, mz, color='#457B9D', lw=1.8, label='$\\langle m_z\\rangle$')
ax1.axhline(0, color='gray', lw=0.5, ls='--')
ax1.set_xlabel('Время (нс)', fontsize=11); ax1.set_ylabel('m', fontsize=11)
ax1.set_title('Динамика намагниченности', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10); ax1.set_ylim(-1.05, 1.05); ax1.grid(True, alpha=0.25)

# ── 2. |m| vs t ──────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gspec[0, 2])
m_abs = np.sqrt(mx**2 + my**2 + mz**2)
ax2.plot(t, m_abs, color='#6A0572', lw=1.8)
ax2.fill_between(t, m_abs, alpha=0.15, color='#6A0572')
ax2.set_xlabel('Время (нс)', fontsize=10); ax2.set_ylabel('|m|', fontsize=10)
ax2.set_title('|m| vs t', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 1.1); ax2.grid(True, alpha=0.25)

# ── 3. Гистограмма размеров ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gspec[1, 0])
bins3 = np.linspace(radii_nm.min()*0.75, radii_nm.max()*1.1, 20)
if len(r_bl): ax3.hist(r_bl, bins=bins3, color='#457B9D', alpha=0.85,
                        label=f'Blocked ({len(r_bl)})', edgecolor='white', lw=0.4)
if len(r_sp): ax3.hist(r_sp, bins=bins3, color='#E63946', alpha=0.85,
                        label=f'SP ({len(r_sp)})', edgecolor='white', lw=0.4)
ax3.axvline(r_crit, color='#F4A261', lw=2, ls='--', label=f'r_crit={r_crit:.1f} нм')
ax3.set_xlabel(f'r ({sim.units.lbl_radius})', fontsize=10)
ax3.set_ylabel('N', fontsize=10)
ax3.set_title('Размеры частиц', fontsize=11, fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.25)

# ── 4. Круговая Blocked/SP ────────────────────────────────────────────────────
ax4 = fig.add_subplot(gspec[1, 1])
n_bl_, n_sp_ = sim._n_blocked, sim._n_sp
total_ = n_bl_ + n_sp_
if total_ > 0:
    ax4.pie(
        [n_bl_, n_sp_],
        labels=[f'Blocked\n{n_bl_} ({100*n_bl_//total_}%)',
                f'SP\n{n_sp_} ({100*n_sp_//total_}%)'],
        colors=['#457B9D', '#E63946'], startangle=90,
        wedgeprops=dict(width=0.55, edgecolor='white', lw=2)
    )
ax4.set_title('Состав ансамбля', fontsize=11, fontweight='bold')

# ── 5. Проекция m (финал) ─────────────────────────────────────────────────────
ax5 = fig.add_subplot(gspec[1, 2])
for p in sim.particles:
    col = '#457B9D' if p.ptype.value == 'blocked' else '#E63946'
    ax5.annotate('', xy=(p.m[0], p.m[2]), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color=col, lw=1.3, alpha=0.8))
circle = plt.Circle((0,0), 1, fill=False, color='lightgray', lw=1.2, ls='--')
ax5.add_patch(circle)
ax5.set_xlim(-1.3,1.3); ax5.set_ylim(-1.3,1.3); ax5.set_aspect('equal')
ax5.set_xlabel('$m_x$', fontsize=10); ax5.set_ylabel('$m_z$', fontsize=10)
ax5.set_title('Финальные m (проекция xz)', fontsize=11, fontweight='bold')
ax5.grid(True, alpha=0.2)
ax5.legend(handles=[
    mpatches.Patch(color='#457B9D', label='Blocked'),
    mpatches.Patch(color='#E63946', label='SP')
], fontsize=9, loc='lower right')

fig.suptitle(
    f'vinamax | {len(sim.particles)} частиц | T={sim.Temp:.0f} K | {sim.units.system}',
    fontsize=14, fontweight='bold'
)
plt.show()

---
## Дополнительно — анализ температурной зависимости r_crit

In [ ]:
# Критический радиус vs температура
# sim.Ku1 уже в СИ внутри объекта
T_arr    = np.linspace(50, 500, 200)
r_crit_arr = np.array([blocking_radius(sim.Ku1, T) * 1e9 for T in T_arr])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(T_arr, r_crit_arr, color='#E63946', lw=2.2)
ax.axhline(R_MEDIAN, color='#457B9D', lw=1.5, ls='--', label=f'r_median = {R_MEDIAN} нм')
ax.axvline(TEMP_K,   color='#F4A261', lw=1.5, ls=':', label=f'T = {TEMP_K} K')
ax.fill_between(T_arr, r_crit_arr, R_MEDIAN,
                where=(r_crit_arr > R_MEDIAN), alpha=0.12, color='#457B9D',
                label='← Blocked зона')
ax.fill_between(T_arr, r_crit_arr, R_MEDIAN,
                where=(r_crit_arr < R_MEDIAN), alpha=0.12, color='#E63946',
                label='← SP зона')
ax.set_xlabel('Температура (К)', fontsize=12)
ax.set_ylabel('r_crit (нм)', fontsize=12)
ax.set_title('Критический радиус блокировки vs T', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Время Нееля vs радиус при разных температурах
r_arr_nm = np.linspace(2, 25, 300)
T_list   = [200, 300, 400]
colors_T = ['#457B9D', '#E63946', '#2A9D8F']

fig, ax = plt.subplots(figsize=(9, 4))
for T_, col in zip(T_list, colors_T):
    tau = [neel_time(sim.Ku1, r*1e-9, T_) for r in r_arr_nm]
    tau = np.clip(tau, 1e-12, 1e15)
    ax.semilogy(r_arr_nm, tau, color=col, lw=2, label=f'T = {T_} K')

ax.axhline(1,    color='gray', lw=1, ls='--', label='τ = 1 с')
ax.axhline(100,  color='gray', lw=1, ls=':',  label='τ = 100 с (τ_obs)')
ax.axvline(R_MEDIAN, color='black', lw=1, ls='--', label=f'r_med = {R_MEDIAN} нм')
ax.set_xlabel('Радиус (нм)', fontsize=12)
ax.set_ylabel('τ_N (с)', fontsize=12)
ax.set_title('Время релаксации Нееля τ_N vs r', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(1e-12, 1e15)
ax.grid(True, alpha=0.3, which='both')
fig.tight_layout()
plt.show()

## Скачать результаты

In [ ]:
files.download('result.txt')